# Comparing boundary value methods in Python

Compare finite differences with Crout or dense LU, shooting, and SciPy's
`solve_bvp`. We solve two problems on $[0,1]$ with
$3y(0)-2y'(0)=0$ and $2y(1)-y'(1)/3=1$:

- Nonlinear: $y''=xyy'-3y^2+6x$, exact solution $y=x^3$.
- Linear: $y''=xy'+3y$, reference from a high-accuracy initial-value solve.

Install the notebook extras and run from the repository root. We import the
same experiment used by the CLI, so the notebook and exported results use one implementation.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root/'examples').exists():
    root = root.parent
sys.path.insert(0, str(root/'examples'))
import pandas as pd
from IPython.display import display
from compare_methods import run_comparison

rows = run_comparison(n=80, repeat=7)
table = pd.DataFrame(rows)
display(table)

,problem,method,nodes,iterations,max_error,median_ms,q25_ms,q75_ms
0,nonlinear,FD + Newton + crout,81,4,7.024513e-05,1.927542,1.877375,2.239854
1,linear,FD + crout,81,1,5.865017e-05,0.275000,0.271333,0.280271
2,nonlinear,FD + Newton + dense,81,4,7.024513e-05,1.546834,1.235541,1.702042
3,linear,FD + dense,81,1,5.865017e-05,0.281000,0.261416,0.330688
4,nonlinear,Newton shooting (RK4),81,4,1.981815e-09,4.183375,3.791229,5.155647
5,linear,Linear shooting (RK4),81,1,1.563087e-09,1.231042,0.843459,1.518854
6,nonlinear,solve_bvp,81,1,1.232185e-11,0.600000,0.509271,0.687500
7,linear,solve_bvp,534,4,2.948197e-13,2.123708,2.006833,2.526667


## Interpret the comparison

Each runtime measures a complete solve, including setup and matrix assembly.
One warm-up is excluded; seven runs give the median and interquartile range.
Plotting, error evaluation and reference construction are outside the timed region.
The CSV command below also saves Python, NumPy, SciPy and platform versions.

Both finite-difference variants use the same equations, starting guess and
Newton tolerances. This isolates their choice of linear solver. Crout has
linear work/storage without pivoting; dense LU uses pivoting and compiled code.
At small sizes, Python loop overhead may dominate Crout, so report measured
times instead of promising a winner.

RK4 shooting has fourth-order integration error; central finite differences
have expected global order two under smoothness and stability. `solve_bvp`
uses adaptive fourth-order collocation and may add nodes. Its tolerance
controls a residual estimate, not the same quantity as Newton's algebraic stopping rule.
**Equal n is not equal work or equal accuracy.** A rigorous efficiency study
would vary each method's settings to reach a common error target.

For the linear reference integrate $u''=xu'+3u$, $u(0)=1$, $u'(0)=3/2$ with
strict tolerances. Set $y=u/[2u(1)-u'(1)/3]$; scaling preserves the homogeneous
left condition and enforces the right one. This is an independent numerical
reference, not a claimed exact solution.

In [2]:
for problem in ['nonlinear', 'linear']:
    subset = table[table.problem == problem].set_index('method')
    crout = subset.loc['FD + Newton + crout' if problem == 'nonlinear' else 'FD + crout']
    dense = subset.loc['FD + Newton + dense' if problem == 'nonlinear' else 'FD + dense']
    print(problem, 'difference in maximum errors:', abs(crout.max_error-dense.max_error))

nonlinear difference in maximum errors: 5.948204168798599e-17
linear difference in maximum errors: 6.994405055138486e-15


## Repeat and export

From the repository root:

```sh
python examples/compare_methods.py --n 80 --repeat 7 --csv results/comparison.csv
```

Try `--n 160` or `--n 320` and record the environment. Repeated measurements
on one computer cannot establish a universal speed ranking. A different
initial guess may also select a different nonlinear branch.

## References

- [Shooting and finite differences](https://metodosnumericos.jonramos.dev/en/articulos/frontera-diferencias-finitas/)
- [SciPy solve_bvp: method and tolerance definitions](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_bvp.html)
- [NumPy dense linear solver](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html)